In [1]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import spacy
import pandas as pd

In [3]:
data = pd.read_csv('bbc_news.csv')

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [5]:
data.head()

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...
2,2,7387,Rationing energy is nothing new for off-grid c...,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,Scoraig in the north west Highlands has long h...
3,3,767,The hunt for superyachts of sanctioned Russian...,"Tue, 22 Mar 2022 14:37:01 GMT",https://www.bbc.co.uk/news/60739336,https://www.bbc.co.uk/news/60739336?at_medium=...,"Wealthy Russians sanctioned by the US, EU and ..."
4,4,3712,Platinum Jubilee: 70 years of the Queen in 70 ...,"Wed, 01 Jun 2022 23:17:33 GMT",https://www.bbc.co.uk/news/uk-61660128,https://www.bbc.co.uk/news/uk-61660128?at_medi...,A quick look back at the Queen's 70 years on t...


In [6]:
titles = pd.DataFrame(data['title'])
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


In [7]:
titles['title_lowercase'] = titles['title'].str.lower()
titles.head()

,title,title_lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...


In [9]:
en_stopwords = stopwords.words('english')
titles['title_no_stopwords'] = titles['title_lowercase'].apply(lambda x: ' '.join([word for word in x.split() if word not in (en_stopwords)]))

In [10]:
titles['title_no_punct'] = titles.apply(lambda x: re.sub(r'[^\w\s]', '', x['title_no_stopwords']), axis = 1)
titles.head()

,title,title_lowercase,title_no_stopwords,title_no_punct
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds


In [11]:
titles['title_tokens'] = titles.apply(lambda x: word_tokenize(x['title']), axis = 1)
titles['title_no_punct_tokens'] = titles.apply(lambda x: word_tokenize(x['title_no_punct']), axis = 1)
titles.head()

,title,title_lowercase,title_no_stopwords,title_no_punct,title_tokens,title_no_punct_tokens
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, offgrid, com..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, 70, years, queen, 70, seco..."


In [16]:
lemmatizer = WordNetLemmatizer()
titles['title_tokens_lemmatized'] = titles['title_tokens'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])
titles['title_no_punct_tokens_lemmatized'] = titles['title_no_punct_tokens'].apply(lambda tokens: [lemmatizer.lemmatize(token) for token in tokens])

In [19]:
final_title_list = sum(titles['title_tokens_lemmatized'], [])
final_title_cleaned_list = sum(titles['title_no_punct_tokens_lemmatized'], [])

In [20]:
print(final_title_list)

['Can', 'I', 'refuse', 'to', 'work', '?', "'Liz", 'Truss', 'the', 'Brief', '?', "'", 'World', 'reacts', 'to', 'UK', 'political', 'turmoil', 'Rationing', 'energy', 'is', 'nothing', 'new', 'for', 'off-grid', 'community', 'The', 'hunt', 'for', 'superyachts', 'of', 'sanctioned', 'Russian', 'oligarch', 'Platinum', 'Jubilee', ':', '70', 'year', 'of', 'the', 'Queen', 'in', '70', 'second', 'Red', 'Bull', 'found', 'guilty', 'of', 'breaking', 'Formula', '1', "'s", 'budget', 'cap', 'World', 'Triathlon', 'Championship', 'Series', ':', 'Flora', 'Duffy', 'beat', 'Georgia', 'Taylor-Brown', 'to', 'woman', "'s", 'title', 'Terry', 'Hall', ':', 'Coventry', 'scooter', 'ride-out', 'pay', 'tribute', 'to', 'singer', 'Post', 'Office', 'and', 'Fujitsu', 'to', 'face', 'inquiry', 'over', 'Horizon', 'scandal', "'Pavement", 'parking', 'frightens', 'me', "'", 'UK', 'interest', 'rate', ':', 'How', 'will', 'the', 'rise', 'affect', 'you', 'and', 'how', 'high', 'could', 'it', 'go', '?', 'They', 'stayed', 'for', 'the', 

In [21]:
print(final_title_cleaned_list)

['refuse', 'work', 'liz', 'truss', 'brief', 'world', 'reacts', 'uk', 'political', 'turmoil', 'rationing', 'energy', 'nothing', 'new', 'offgrid', 'community', 'hunt', 'superyachts', 'sanctioned', 'russian', 'oligarch', 'platinum', 'jubilee', '70', 'year', 'queen', '70', 'second', 'red', 'bull', 'found', 'guilty', 'breaking', 'formula', '1', 'budget', 'cap', 'world', 'triathlon', 'championship', 'series', 'flora', 'duffy', 'beat', 'georgia', 'taylorbrown', 'womens', 'title', 'terry', 'hall', 'coventry', 'scooter', 'rideout', 'pay', 'tribute', 'singer', 'post', 'office', 'fujitsu', 'face', 'inquiry', 'horizon', 'scandal', 'pavement', 'parking', 'frightens', 'me', 'uk', 'interest', 'rate', 'rise', 'affect', 'high', 'could', 'go', 'stayed', 'storm', 'happens', 'now', 'six', 'nation', 'scotland', 'best', 'since', '99', 'beat', 'best', 'ireland', 'ever', 'long', 'liz', 'truss', 'survive', 'prime', 'minister', 'platinum', 'jubilee', 'beacon', 'light', 'across', 'globe', 'celebrate', 'queen', '

In [22]:
nlp = spacy.load('en_core_web_sm')

In [23]:
spacy_docs_final_title = nlp(' '.join(final_title_list))
print(spacy_docs_final_title)

Can I refuse to work ? 'Liz Truss the Brief ? ' World reacts to UK political turmoil Rationing energy is nothing new for off-grid community The hunt for superyachts of sanctioned Russian oligarch Platinum Jubilee : 70 year of the Queen in 70 second Red Bull found guilty of breaking Formula 1 's budget cap World Triathlon Championship Series : Flora Duffy beat Georgia Taylor-Brown to woman 's title Terry Hall : Coventry scooter ride-out pay tribute to singer Post Office and Fujitsu to face inquiry over Horizon scandal 'Pavement parking frightens me ' UK interest rate : How will the rise affect you and how high could it go ? They stayed for the storm - what happens now ? Six Nations : Can Scotland 's best since '99 beat best Ireland ever ? How long can Liz Truss survive a prime minister ? Platinum Jubilee : Beacons light across the globe to celebrate Queen 's reign Top student loan interest rate cut by 5 % in England Russians said they ’ d take my baby : A medic ’ s story Week in picture

In [24]:
spacy_docs_final_title_cleaned = nlp(' '.join(final_title_cleaned_list))
print(spacy_docs_final_title_cleaned)

refuse work liz truss brief world reacts uk political turmoil rationing energy nothing new offgrid community hunt superyachts sanctioned russian oligarch platinum jubilee 70 year queen 70 second red bull found guilty breaking formula 1 budget cap world triathlon championship series flora duffy beat georgia taylorbrown womens title terry hall coventry scooter rideout pay tribute singer post office fujitsu face inquiry horizon scandal pavement parking frightens me uk interest rate rise affect high could go stayed storm happens now six nation scotland best since 99 beat best ireland ever long liz truss survive prime minister platinum jubilee beacon light across globe celebrate queen reign top student loan interest rate cut 5 england russian said theyd take baby medic story week picture 1117 june 2022 xi jinping hong kong tight security city mark 25 year handover rescue carried silence listen sound life nick pope newcastle goalkeeper extends incredible clean sheet record china plan become 

In [28]:
pos_docs_rt = pd.DataFrame(columns=['tokens_rt', 'pos_tags_rt'])
pos_docs_cl = pd.DataFrame(columns=['tokens_cl', 'pos_tags_cl'])

In [30]:
for word in spacy_docs_final_title:
    pos_docs_rt = pd.concat([pos_docs_rt, pd.DataFrame.from_records([{'tokens_rt': word.text, 'pos_tags_rt': word.pos_}])], ignore_index=True)    
pos_docs_rt.head()    

,tokens_rt,pos_tags_rt
0,Can,AUX
1,I,PRON
2,refuse,VERB
3,to,PART
4,work,VERB


In [31]:
for word in spacy_docs_final_title_cleaned:
    pos_docs_cl = pd.concat([pos_docs_cl, pd.DataFrame.from_records([{'tokens_cl': word.text, 'pos_tags_cl': word.pos_}])], ignore_index=True)
pos_docs_cl.head()

,tokens_cl,pos_tags_cl
0,refuse,AUX
1,work,NOUN
2,liz,PROPN
3,truss,ADJ
4,brief,ADJ


In [33]:
pos_docs_rt_counts = pos_docs_rt.groupby(['tokens_rt', 'pos_tags_rt']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)
pos_docs_rt_counts.head(10)

,tokens_rt,pos_tags_rt,counts
95,:,PUNCT,543
8,',PUNCT,314
2805,in,ADP,187
3875,to,PART,174
3136,of,ADP,172
22,-,PUNCT,166
3839,the,DET,163
1857,and,CCONJ,147
15,'s,PART,143
97,?,PUNCT,130


In [36]:
pos_docs_cl_counts = pos_docs_cl.groupby(['tokens_cl', 'pos_tags_cl']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)
pos_docs_cl_counts.head(10)

,tokens_cl,pos_tags_cl,counts
30,2022,NUM,47
1162,england,PROPN,45
870,cup,PROPN,39
3056,say,VERB,37
3707,uk,PROPN,37
3840,war,NOUN,34
2386,new,ADJ,31
3948,world,NOUN,30
3949,world,PROPN,26
3710,ukraine,PROPN,23


In [38]:
propn = pos_docs_cl_counts[pos_docs_cl_counts.pos_tags_cl == 'PROPN']
propn

,tokens_cl,pos_tags_cl,counts
1162,england,PROPN,45
870,cup,PROPN,39
3707,uk,PROPN,37
3949,world,PROPN,26
3710,ukraine,PROPN,23
...,...,...,...
1553,hail,PROPN,1
1555,hale,PROPN,1
1556,haley,PROPN,1
1562,halloween,PROPN,1


In [39]:
verb = pos_docs_cl_counts[pos_docs_cl_counts.pos_tags_cl == 'VERB']
verb

,tokens_cl,pos_tags_cl,counts
3056,say,VERB,37
3711,ukraine,VERB,22
358,beat,VERB,13
2133,make,VERB,13
3461,take,VERB,13
...,...,...,...
1583,hardline,VERB,1
1598,heading,VERB,1
1549,hacked,VERB,1
1550,hadwin,VERB,1


In [40]:
ner_docs_cl = pd.DataFrame(columns=['tokens_cl', 'ner_cl'])

In [46]:
for token in spacy_docs_final_title_cleaned.ents:
    if pd.isna(token.label_) is False:
        ner_docs_cl = pd.concat([ner_docs_cl, pd.DataFrame.from_records([{'tokens_cl': token.text,'ner_cl': token.label_}])], ignore_index=True)
ner_docs_cl.head(10)    

,tokens_cl,ner_cl
0,russian,NORP
1,70 year,DATE
2,70 second,TIME
3,bull,ORG
4,1,CARDINAL
5,georgia taylorbrown womens,ORG
6,terry hall,PERSON
7,six,CARDINAL
8,99,CARDINAL
9,jubilee beacon,PERSON
